# SMS Spam Classification using RNN and LSTM

**Module:** 6CS012 Artificial Intelligence and Machine Learning
**Student:** Suyank Hang Rai | **ID:** 2331514
**Group:** L6CG20
**Tutor:** Ms. Durga Pokharel

This notebook covers Part III (Language Tasks) of the Final Portfolio Assessment.

**Dataset:** SMS Spam Collection v.1 binary classification: Spam vs Ham
**Models:** Simple RNN, LSTM with trainable embeddings, LSTM with GloVe pretrained embeddings
**Bonus:** Real-time prediction interface using Gradio

## 0. Install Dependencies

Run this cell first. Gensim is not preinstalled in Colab. The numpy pin (1.23.5) is required for gensim compatibility.

In [ ]:
!pip install numpy==1.23.5 --quiet
!pip install gensim --quiet
!pip install wordcloud --quiet
!pip install nltk --quiet
print('All packages installed.')

## 1. Imports and Setup

All imports collected here. A fixed random seed is set for reproducibility.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, string, time, warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from wordcloud import WordCloud
import gensim.downloader as api

tf.random.set_seed(42)
np.random.seed(42)

print('All imports successful.')
print(f'TensorFlow version: {tf.__version__}')

## 2. Load Dataset

The SMS Spam Collection v.1 is loaded. Columns are renamed for clarity: v1 becomes label, v2 becomes text.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Ai/text classification/spamvsham.csv', encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'text']

print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## 3. Exploratory Data Analysis

Checking class balance, message length distribution, and most common words. Knowing the data is essential for choosing preprocessing steps and interpreting model performance later.

In [ ]:
print('=== Dataset Overview ===')
print(f'Total messages  : {len(df)}')
print(f'Ham  (not spam) : {(df.label=="ham").sum()} ({(df.label=="ham").mean()*100:.1f}%)')
print(f'Spam            : {(df.label=="spam").sum()} ({(df.label=="spam").mean()*100:.1f}%)')
print(f'Null values     : {df.isnull().sum().sum()}')
print(f'Duplicate rows  : {df.duplicated().sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts()

axes[0].bar(counts.index, counts.values, color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label'); axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.suptitle('SMS Spam Collection Class Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nClass imbalance ratio: {counts.max() / counts.min():.1f}x')

In [ ]:
df['msg_len'] = df['text'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, lbl, color in zip(axes, ['ham', 'spam'], ['steelblue', 'tomato']):
    subset = df[df['label'] == lbl]['msg_len']
    ax.hist(subset, bins=40, color=color, edgecolor='black', alpha=0.85)
    ax.set_title(f'Message Length {lbl.capitalize()}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Character Count'); ax.set_ylabel('Frequency')
    ax.axvline(subset.mean(), color='black', linestyle='--', label=f'Mean: {subset.mean():.0f}')
    ax.legend()
plt.tight_layout()
plt.show()

print('\nMessage Length Statistics:')
print(df.groupby('label')['msg_len'].describe().round(1))

## 4. Text Preprocessing and Cleaning

Raw SMS text is noisy. Each step addresses a specific problem:

Lowercase ensures "Free" and "free" are treated the same.
Contraction expansion converts "don't" to "do not".
URL, mention, and hashtag removal strips noise with no semantic content.
Number removal eliminates raw digits that are not useful features.
Punctuation removal simplifies the token vocabulary.
Stopword removal eliminates high-frequency low-signal words.
Lemmatisation reduces inflected forms to their base form to reduce vocabulary size.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

CONTRACTIONS = {
    "don't": "do not", "won't": "will not", "can't": "cannot",
    "i'm": "i am", "i've": "i have", "it's": "it is",
    "i'll": "i will", "i'd": "i would", "you're": "you are",
    "you've": "you have", "you'll": "you will", "that's": "that is",
    "there's": "there is", "we're": "we are", "we've": "we have",
    "we'll": "we will", "they're": "they are", "they've": "they have",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "hasn't": "has not", "haven't": "have not",
    "hadn't": "had not", "doesn't": "does not", "didn't": "did not",
    "couldn't": "could not", "wouldn't": "would not", "shouldn't": "should not",
}

def clean_text(text):
    text = text.lower()
    for k, v in CONTRACTIONS.items():
        text = text.replace(k, v)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens
              if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(clean_text)

print('Sample before/after cleaning:')
for i in [2, 3, 10]:
    print(f'\n[{df.label.iloc[i].upper()}]')
    print(f'  Original : {df.text.iloc[i][:120]}')
    print(f'  Cleaned  : {df.clean_text.iloc[i][:120]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, lbl, cmap in zip(axes, ['ham', 'spam'], ['Blues', 'Reds']):
    corpus = ' '.join(df[df['label'] == lbl]['clean_text'])
    wc = WordCloud(width=600, height=350, background_color='white',
                   colormap=cmap, max_words=100, collocations=False).generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Word Cloud {lbl.capitalize()}', fontsize=14, fontweight='bold')
plt.suptitle('Most Frequent Words per Class', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, lbl, color in zip(axes, ['ham', 'spam'], ['steelblue', 'tomato']):
    words = ' '.join(df[df['label'] == lbl]['clean_text']).split()
    top15 = Counter(words).most_common(15)
    words_list, counts_list = zip(*top15)
    ax.barh(words_list[::-1], counts_list[::-1], color=color, edgecolor='black')
    ax.set_title(f'Top 15 Words {lbl.capitalize()}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Frequency')
plt.tight_layout()
plt.show()

## 5. Tokenisation, Encoding, and Padding

The cleaned text needs to be converted to numbers before feeding to a neural network:

1. Label encoding: spam=1, ham=0 (binary classification)
2. Train/test split: stratified 80/20 split to preserve class proportions
3. Tokenisation: each unique word gets an integer ID, vocabulary capped at 10000 most frequent words
4. Percentile-based padding: sequences padded/truncated to 95th percentile length, avoiding inflated sequence lengths from outlier messages

In [ ]:
df['label_enc'] = (df['label'] == 'spam').astype(int)
print('Label encoding: ham=0, spam=1')
print(df['label_enc'].value_counts())

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_text'], df['label_enc'],
    test_size=0.20, random_state=42, stratify=df['label_enc']
)
print(f'\nTrain: {len(X_train_raw)} | Test: {len(X_test_raw)}')
print(f'Train spam %: {y_train.mean()*100:.1f}% | Test spam %: {y_test.mean()*100:.1f}%')

In [ ]:
MAX_VOCAB = 10000
OOV_TOKEN = '<OOV>'

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train_raw)

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
print(f'Vocabulary size (capped at {MAX_VOCAB}): {vocab_size}')
print(f'Total unique tokens in training set   : {len(tokenizer.word_index)}')

X_train_seq = tokenizer.texts_to_sequences(X_train_raw)
X_test_seq  = tokenizer.texts_to_sequences(X_test_raw)

In [ ]:
train_lengths = [len(s) for s in X_train_seq]
MAX_LEN = int(np.percentile(train_lengths, 95))
print(f'95th percentile sequence length: {MAX_LEN}')
print(f'Mean: {np.mean(train_lengths):.1f}  |  Max: {max(train_lengths)}')

plt.figure(figsize=(8, 4))
plt.hist(train_lengths, bins=40, color='steelblue', edgecolor='black')
plt.axvline(MAX_LEN, color='red', linestyle='--', label=f'95th pct = {MAX_LEN}')
plt.title('Training Sequence Length Distribution', fontweight='bold')
plt.xlabel('Sequence Length (tokens)'); plt.ylabel('Frequency')
plt.legend(); plt.tight_layout()
plt.show()

X_train = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')
y_train = np.array(y_train)
y_test  = np.array(y_test)

print(f'\nX_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')

## 6. Model Building

Three models built with increasing complexity:

1. **Simple RNN** with trainable Embedding layer: fast but struggles with longer sequences due to the vanishing gradient problem.
2. **LSTM** with trainable Embedding layer: gated cell structure that explicitly manages long-range dependencies.
3. **LSTM with pretrained GloVe embeddings**: uses pretrained 50-dimensional GloVe embeddings, bringing in external semantic knowledge.

In [ ]:
EMBEDDING_DIM = 64
EPOCHS        = 15
BATCH_SIZE    = 64

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, ylabel in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
        ax.plot(history.history[metric],          label='Train', linewidth=2)
        ax.plot(history.history[f'val_{metric}'], label='Val',   linewidth=2, linestyle='--')
        ax.set_title(f'{title} {ylabel}', fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.legend()
    plt.tight_layout()
    plt.show()

def evaluate_model(model, X_test, y_test, name):
    y_pred_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_pred_prob >= 0.5).astype(int)
    acc = accuracy_score(y_test, y_pred)
    print(f'\n=== {name} ===')
    print(f'Accuracy: {acc*100:.2f}%')
    print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
    plt.title(f'Confusion Matrix {name}', fontweight='bold')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()
    return acc, y_pred

print('Helpers defined.')

### Model 1: Simple RNN with Trainable Embedding

A single SimpleRNN layer with 64 units. The embedding layer starts with random weights and learns word representations during training. This is the simplest baseline for sequential text data.

**Limitation:** SimpleRNN struggles with sequences longer than roughly 20 tokens because gradients vanish as they are backpropagated through many time steps.

In [ ]:
model1 = Sequential(name='SimpleRNN_Model', layers=[
    Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    SimpleRNN(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model1.summary()

t0 = time.time()
history1 = model1.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)
time_model1 = time.time() - t0
print(f'\nModel 1 training time: {time_model1:.1f}s')

plot_history(history1, 'Model 1 Simple RNN')
acc1, pred1 = evaluate_model(model1, X_test, y_test, 'Model 1 Simple RNN')

### Model 2: LSTM with Trainable Embedding

Two stacked LSTM layers (64 units then 32 units). Stacking allows the second LSTM to operate on the sequence of hidden states produced by the first, learning higher-order temporal patterns.

**Why LSTM over RNN?** LSTM cells have three gates (input, forget, output) that explicitly control how much information passes through at each time step. This solves the vanishing gradient problem for sequences up to a few hundred tokens.

In [ ]:
model2 = Sequential(name='LSTM_Model', layers=[
    Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model2.summary()

t0 = time.time()
history2 = model2.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)
time_model2 = time.time() - t0
print(f'\nModel 2 training time: {time_model2:.1f}s')

plot_history(history2, 'Model 2 LSTM')
acc2, pred2 = evaluate_model(model2, X_test, y_test, 'Model 2 LSTM')

### Model 3: LSTM with Pretrained GloVe Embeddings

This model replaces the randomly initialised embedding layer with pretrained GloVe vectors (50 dimensions, trained on Wikipedia and Gigaword). The embedding layer weights are frozen (trainable=False) so GloVe is used purely as a feature extractor.

**Why GloVe?** The SMS training set has only around 4500 messages. A randomly initialised embedding layer needs to learn word representations from this small corpus, which is insufficient for rare words. GloVe brings in knowledge from billions of tokens.

In [ ]:
print('Downloading GloVe embeddings...')
glove_model = api.load('glove-wiki-gigaword-50')
GLOVE_DIM = 50
print('GloVe embeddings loaded.')

In [ ]:
word_index = tokenizer.word_index
embedding_matrix = np.zeros((vocab_size, GLOVE_DIM))

found = 0
for word, idx in word_index.items():
    if idx >= vocab_size:
        continue
    if word in glove_model:
        embedding_matrix[idx] = glove_model[word]
        found += 1

coverage = found / min(len(word_index), vocab_size) * 100
print(f'Words found in GloVe: {found} / {min(len(word_index), vocab_size)} ({coverage:.1f}% coverage)')

In [ ]:
model3 = Sequential(name='LSTM_GloVe_Model', layers=[
    Embedding(input_dim=vocab_size,
              output_dim=GLOVE_DIM,
              weights=[embedding_matrix],
              input_length=MAX_LEN,
              trainable=False),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model3.summary()

t0 = time.time()
history3 = model3.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)
time_model3 = time.time() - t0
print(f'\nModel 3 training time: {time_model3:.1f}s')

plot_history(history3, 'Model 3 LSTM GloVe')
acc3, pred3 = evaluate_model(model3, X_test, y_test, 'Model 3 LSTM GloVe')

## 7. Model Comparison

Bringing all three models together for a direct comparison on accuracy and training time.

In [ ]:
model_names = ['Simple RNN', 'LSTM', 'LSTM + GloVe']
accuracies  = [acc1*100, acc2*100, acc3*100]
train_times = [time_model1, time_model2, time_model3]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bars = axes[0].bar(model_names, accuracies, color=['steelblue', 'seagreen', 'tomato'], edgecolor='black')
axes[0].set_ylim(90, 100)
axes[0].set_title('Test Accuracy Comparison', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Accuracy (%)')
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{acc:.2f}%', ha='center', fontweight='bold')

axes[1].bar(model_names, train_times, color=['steelblue', 'seagreen', 'tomato'], edgecolor='black')
axes[1].set_title('Training Time Comparison', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Seconds')
for i, t in enumerate(train_times):
    axes[1].text(i, t + 0.5, f'{t:.1f}s', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

summary_df = pd.DataFrame({
    'Model': model_names,
    'Accuracy (%)': [f'{a:.2f}' for a in accuracies],
    'Training Time (s)': [f'{t:.1f}' for t in train_times]
})
print(summary_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'seagreen', 'tomato']
histories = [history1, history2, history3]
names_short = ['Simple RNN', 'LSTM', 'LSTM+GloVe']

for hist, name, color in zip(histories, names_short, colors):
    epochs_range = range(1, len(hist.history['accuracy']) + 1)
    axes[0].plot(epochs_range, hist.history['val_accuracy'], label=name, color=color, linewidth=2)
    axes[1].plot(epochs_range, hist.history['val_loss'],     label=name, color=color, linewidth=2)

axes[0].set_title('Validation Accuracy All Models', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

axes[1].set_title('Validation Loss All Models', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Error Analysis

Using LSTM (Model 2) as the primary model for error analysis.

Misclassifications fall into two categories:
**False negatives (spam predicted as ham):** The model missed spam, which reaches the user inbox.
**False positives (ham predicted as spam):** Legitimate messages filtered incorrectly.

Understanding which examples the model gets wrong reveals systematic weaknesses.

In [ ]:
X_test_raw_list = list(X_test_raw)
y_test_list     = list(y_test)
y_pred_list     = list(pred2)

errors = [(X_test_raw_list[i], y_test_list[i], y_pred_list[i])
          for i in range(len(y_test_list))
          if y_test_list[i] != y_pred_list[i]]

print(f'Total misclassifications (LSTM): {len(errors)} / {len(y_test)}')
print(f'Error rate: {len(errors)/len(y_test)*100:.2f}%\n')

fn = [(t, l, p) for t, l, p in errors if l == 1]
fp = [(t, l, p) for t, l, p in errors if l == 0]
print(f'False Negatives (spam missed): {len(fn)}')
print(f'False Positives (ham misfiled): {len(fp)}')

label_map = {0: 'Ham', 1: 'Spam'}
print('\n=== Sample Misclassifications ===')
for i, (txt, true, pred) in enumerate(errors[:5]):
    print(f'\n[Example {i+1}]')
    print(f'  Text      : {txt[:120]}')
    print(f'  True Label: {label_map[true]}')
    print(f'  Predicted : {label_map[pred]}')
    if true == 1:
        print('  False Negative: spam evaded filter (informal or context-dependent language)')
    else:
        print('  False Positive: ham flagged (urgency words without spammy intent)')

**Analysis of misclassifications:**

False negatives tend to involve spam using informal conversational language to bypass filters without ALL CAPS or FREE patterns. False positives usually contain words that appear in spam (like call, free, urgent) but in a legitimate context. Both error types point to the same root cause: the model relies heavily on individual word signals rather than full contextual meaning.

**Potential improvements:**
1. Class weighting to penalise false negatives more heavily
2. Bidirectional LSTM to capture context from both directions
3. Character-level features to catch obfuscated spam
4. BERT fine-tuning for richer contextual representations

## 9. Real-Time GUI with Gradio

A simple prediction interface using Gradio. The user types an SMS message and gets a spam/ham prediction with confidence score. Uses Model 2 (LSTM) as the prediction backend.

In [ ]:
!pip install gradio --quiet
print('Gradio installed.')

In [ ]:
import gradio as gr

def predict_spam(message: str) -> str:
    cleaned  = clean_text(message)
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded   = pad_sequences(sequence, maxlen=MAX_LEN, padding='post', truncating='post')
    prob     = model2.predict(padded, verbose=0)[0][0]
    label    = 'SPAM' if prob >= 0.5 else 'HAM'
    return f'{label}  (spam probability: {prob*100:.1f}%)'

examples = [
    ["WINNER!! You have been selected for a $1000 prize. Call now!"],
    ["Hey, are you coming to the study group tonight?"],
    ["FREE entry into our weekly competition! Text WIN to 87070."],
    ["Can you pick up some groceries on your way home?"],
    ["Congratulations! Your mobile has been awarded a free ringtone."],
]

demo = gr.Interface(
    fn=predict_spam,
    inputs=gr.Textbox(lines=3, placeholder='Type an SMS message here...', label='SMS Message'),
    outputs=gr.Textbox(label='Prediction'),
    title='SMS Spam Detector',
    description='Classifies an SMS message as Spam or Ham using a stacked LSTM model trained on the SMS Spam Collection dataset.',
    examples=examples,
    theme='soft'
)

demo.launch(share=True)

## 10. Discussion and Observations

### Dataset
The SMS Spam Collection v.1 contains 5574 messages: 4827 ham (86.6 percent) and 747 spam (13.4 percent). The class imbalance means accuracy is a misleading metric. A model predicting ham every time gets 86.6 percent accuracy. Precision and recall for the spam class are the more meaningful measures.

### Preprocessing
Each step in the cleaning pipeline contributed to a cleaner vocabulary. Contraction expansion is especially useful as won't and will not should be treated as the same concept. Lemmatisation reduces the vocabulary by collapsing inflected forms, which is important when training data is limited.

### Model Comparison

Simple RNN: fast with limited long-range memory due to no gating mechanism.
LSTM: handles sequence dependencies better through its gated cell structure.
LSTM with GloVe: richer word representations from pre-trained embeddings require fewer updates.

LSTMs generally outperform Simple RNNs on text because SMS spam detection relies on word order and phrase-level patterns that a plain RNN forgets quickly.

### GloVe vs Trainable Embeddings
GloVe embeddings bring in external semantic knowledge. Words that are rare in the 5574-message dataset already have reasonable vector representations. The tradeoff is that frozen embeddings cannot adapt to SMS-specific informal language.

### Limitations
Small dataset: 5574 messages is manageable but limits generalisation.
Binary labels only: no neutral or phishing category.
GloVe coverage is imperfect for informal SMS language with abbreviations and slang.

### Future Work
1. Bidirectional LSTM for better contextual understanding
2. Class weighting to reduce false negatives specifically
3. BERT or DistilBERT fine-tuning for transformer-based attention
4. Character-level features to catch obfuscated spam